# 04 Simulador de clasificación MGCECDL

Simulador interactivo para evaluar cómo cambios interpretables en variables de entrada modifican las probabilidades de clase del modelo de clasificación. El modo inicial usa las variables priorizadas por el cotejo, pero el usuario puede cambiar a todas las variables disponibles del modelo.


## 1. Importaciones y rutas


In [7]:
%load_ext autoreload
%autoreload 2

import io
import os
import sys
import warnings
from contextlib import redirect_stdout
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError("Este cuaderno requiere ipywidgets para la interfaz interactiva.") from exc

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from chec_impacto.data import preparar_splits_estratificados, procesar_dataset_completo
from chec_impacto.training import (
    cargar_modelo_mgcecdl,
    escalar_features_minmax_mgcecdl,
    predict_classification,
    resolve_training_device,
)
from chec_local_interpreter.simulator import (
    default_simulation_values,
    read_prioritized_variables_table,
    safe_plot_filename,
    save_prioritized_variables_table,
    simulator_plots_dir,
    simulator_results_path,
    prioritized_variables_path,
    simulate_feature_class_transitions,
    validate_prioritized_variables,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Carga de modelo, datos y preprocesamiento


In [8]:
DATA_PATH = ROOT / "data" / "Indicadores_vano_v3.csv"
VARIABLES_SELECCION_PATH = ROOT / "data" / "Variables_seleccion.xlsx"
MODEL_DIR = ROOT / "data" / "models"
OUTPUT_DIR = ROOT / "reports" / "interpretability" / "artifacts"
PRIORITIZED_VARIABLES_PATH = prioritized_variables_path(OUTPUT_DIR)
SIMULATOR_RESULTS_PATH = simulator_results_path(OUTPUT_DIR)
SIMULATOR_PLOTS_DIR = simulator_plots_dir(OUTPUT_DIR)

FILTRO_UITI_MAX = None
VENTANA_CLIMATICA_HORAS = 12
SHAP_RANDOM_STATE = 42
DEVICE = resolve_training_device("auto")


def modelo_mas_reciente(model_dir: Path, base_name: str) -> Path:
    stem = Path(base_name).stem
    suffix = Path(base_name).suffix
    candidates = sorted(model_dir.glob(f"{stem}*{suffix}"))
    if not candidates:
        raise FileNotFoundError(f"No encontrado: {base_name} en {model_dir}")
    return candidates[-1]

MODEL_PATH = modelo_mas_reciente(MODEL_DIR, "mgcecdl_classifier_best.zip")

with redirect_stdout(io.StringIO()):
    datos = procesar_dataset_completo(
        path_clima=DATA_PATH,
        path_variables_seleccion=VARIABLES_SELECCION_PATH,
        use_sampling=False,
        min_samples_per_codigo=5,
        target="UITI_VANO",
        filtro_uiti_max=FILTRO_UITI_MAX,
        ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
    )

feature_names = list(datos["features"])
X_raw_model = np.asarray(datos["X"], dtype=np.float32)
Xdf = datos["Xdata"].copy().reset_index(drop=True)
context_df = datos["df_original_copy"].copy().reset_index(drop=True)
label_encoders = datos.get("label_encoders", {})
max_values_imputed = datos.get("max_values_imputed", {})

splits_clf = escalar_features_minmax_mgcecdl(
    preparar_splits_estratificados(
        X_raw_model,
        datos["y"],
        modo="clasificacion",
        random_state=SHAP_RANDOM_STATE,
    )
)
feature_scaler = splits_clf["feature_scaler"]
X = feature_scaler.transform(X_raw_model).astype(np.float32)

modelo = cargar_modelo_mgcecdl(str(MODEL_PATH), device=DEVICE)
probe = predict_classification(modelo, X[: min(len(X), 256)], device=DEVICE)
n_classes = int(np.asarray(probe["fused_probs"]).shape[1])
def risk_class_labels(n_classes):
    if n_classes == 4:
        return [
            "Riesgo bajo (Q1)",
            "Riesgo medio-bajo (Q2)",
            "Riesgo medio-alto (Q3)",
            "Riesgo alto (Q4)",
        ]
    if n_classes == 3:
        return ["Riesgo bajo", "Riesgo medio", "Riesgo alto"]
    if n_classes == 2:
        return ["Riesgo bajo", "Riesgo alto"]
    return [f"Riesgo ordinal {idx}" for idx in range(n_classes)]

class_labels = risk_class_labels(n_classes)

baseline_outputs = predict_classification(modelo, X, device=DEVICE)
baseline_predicted_classes = np.asarray(baseline_outputs["predicted_classes"], dtype=int)
baseline_probabilities = np.asarray(baseline_outputs["fused_probs"], dtype=float)



Dataset original: X=(159470, 70), y=(159470, 1)
Splits generados -> Train: (102060, 70), Valid: (25516, 70), Test: (31894, 70)
Modo objetivo: clasificacion

Distribución de clases para estratificación:
Original: [39868 39867 39867 39868]
Train:    [25515 25515 25515 25515]
Valid:    [6379 6379 6379 6379]
Test:     [7974 7973 7973 7974]


## 3. Carga y validación de variables priorizadas


In [9]:
prioritized_df, prioritized_warnings = read_prioritized_variables_table(PRIORITIZED_VARIABLES_PATH)
for message in prioritized_warnings:
    warnings.warn(message, RuntimeWarning)

variables_priorizadas_disponibles, validation_warnings = validate_prioritized_variables(prioritized_df, feature_names)
for message in validation_warnings:
    warnings.warn(message, RuntimeWarning)

variables_modelo_disponibles = [name for name in feature_names if name in Xdf.columns]
if not variables_modelo_disponibles:
    raise RuntimeError("No hay variables del modelo con valores originales disponibles en Xdf.")

if not variables_priorizadas_disponibles:
    warnings.warn(
        "No hay variables priorizadas válidas; el simulador inicia con todas las variables disponibles.",
        RuntimeWarning,
    )


def circuito_interes_desde_variables_priorizadas(prioritized_df, context_df):
    if "circuito" not in prioritized_df.columns or "CIRCUITO" not in context_df.columns:
        return "Todos"
    candidatos = prioritized_df["circuito"].dropna().astype(str).str.strip()
    candidatos = candidatos[candidatos.ne("")]
    if candidatos.empty:
        return "Todos"
    circuitos_disponibles = set(context_df["CIRCUITO"].dropna().astype(str))
    for candidato in candidatos:
        if candidato in circuitos_disponibles:
            return candidato
    return "Todos"


circuito_interes_default = circuito_interes_desde_variables_priorizadas(prioritized_df, context_df)



## 4. Funciones auxiliares del simulador


In [10]:
def variable_options_for_mode(mode):
    if mode == "Variables priorizadas" and variables_priorizadas_disponibles:
        return variables_priorizadas_disponibles
    return variables_modelo_disponibles


def sorted_text_values(series):
    return sorted(series.dropna().astype(str).unique().tolist())


def fechas_contexto():
    if "FECHA" not in context_df.columns:
        return pd.Series(pd.NaT, index=context_df.index)
    return pd.to_datetime(context_df["FECHA"], errors="coerce")


context_dates = fechas_contexto()
valid_context_dates = context_dates.dropna()
default_fecha_inicio = valid_context_dates.min().date() if not valid_context_dates.empty else None
default_fecha_fin = valid_context_dates.max().date() if not valid_context_dates.empty else None


def filter_mask(circuito="Todos", fid_vano="Todos", fecha_inicio=None, fecha_fin=None):
    mask = pd.Series(True, index=context_df.index)
    if circuito not in (None, "", "Todos") and "CIRCUITO" in context_df.columns:
        mask &= context_df["CIRCUITO"].astype(str).eq(str(circuito))
    if fid_vano not in (None, "", "Todos") and "FID_VANO" in context_df.columns:
        mask &= context_df["FID_VANO"].astype(str).eq(str(fid_vano))
    if not valid_context_dates.empty:
        if fecha_inicio is not None:
            mask &= context_dates.ge(pd.Timestamp(fecha_inicio))
        if fecha_fin is not None:
            mask &= context_dates.le(pd.Timestamp(fecha_fin))
    return mask.to_numpy(dtype=bool)


def circuit_options():
    if "CIRCUITO" not in context_df.columns:
        return ["Todos"]
    return ["Todos"] + sorted_text_values(context_df["CIRCUITO"])


def vano_options_for_circuit(circuito="Todos"):
    if "FID_VANO" not in context_df.columns:
        return ["Todos"]
    if circuito not in (None, "", "Todos") and "CIRCUITO" in context_df.columns:
        mask = context_df["CIRCUITO"].astype(str).eq(str(circuito))
        values = sorted_text_values(context_df.loc[mask, "FID_VANO"])
    else:
        values = sorted_text_values(context_df["FID_VANO"])
    return ["Todos"] + values


def is_categorical_variable(variable):
    if variable in label_encoders:
        return True
    return not pd.api.types.is_numeric_dtype(Xdf[variable])


def categorical_values_for_variable(variable, max_values=None):
    if variable in label_encoders:
        values = [str(value) for value in label_encoders[variable].classes_]
    else:
        values = sorted_text_values(Xdf[variable])
    values = values or ["no aplica"]
    return values if max_values is None else values[:max_values]


def make_value_widget(variable):
    if is_categorical_variable(variable):
        values = categorical_values_for_variable(variable)
        text = f"Categorica: se evaluaran automaticamente {len(values)} categorias."
        return widgets.HTML(value=f"<b>Valor</b>: {text}")

    series = Xdf[variable]
    numeric = pd.to_numeric(series, errors="coerce").dropna()
    if numeric.empty:
        return widgets.Text(value="", description="Valor")
    min_value = float(numeric.min())
    max_value = float(numeric.max())
    median_value = float(numeric.median())
    if np.isclose(min_value, max_value):
        return widgets.FloatText(value=median_value, description="Valor")
    step = max((max_value - min_value) / 100.0, 1e-6)
    return widgets.FloatSlider(
        value=median_value,
        min=min_value,
        max=max_value,
        step=step,
        description="Valor",
        continuous_update=False,
        readout_format=".4g",
    )


def values_grid_for_variable(variable, selected_value=None, max_values=18):
    if is_categorical_variable(variable):
        return categorical_values_for_variable(variable)
    values = default_simulation_values(Xdf[variable], max_values=max_values)
    if selected_value not in (None, "") and selected_value not in values:
        values.append(selected_value)
    return values



def save_simulation_outputs(result_df, variable):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    SIMULATOR_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    result_df.to_excel(SIMULATOR_RESULTS_PATH, index=False)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    plot_df = result_df.copy()
    try:
        plot_df["_x"] = pd.to_numeric(plot_df["valor_original"], errors="raise")
        plot_df = plot_df.sort_values("_x")
        x_values = plot_df["_x"]
        ax.set_xlabel(variable)
    except Exception:
        plot_df["_x"] = plot_df["valor_original"].astype(str)
        x_values = plot_df["_x"]
        ax.set_xlabel(variable)
        ax.tick_params(axis="x", rotation=45)

    for class_idx, label in enumerate(class_labels):
        column = f"prob_clase_{class_idx}_promedio"
        if column in plot_df.columns:
            ax.plot(x_values, plot_df[column], marker="o", linewidth=2, label=label)

    ax.set_ylabel("Probabilidad softmax promedio")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
    ax.set_title(f"Softmax por clase según {variable}")
    fig.tight_layout()
    pdf_path = SIMULATOR_PLOTS_DIR / safe_plot_filename(variable)
    png_path = pdf_path.with_suffix(".png")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    return fig, png_path

def interactive_classification_feature_simulator():
    clear_output(wait=True)
    global _SIMULATOR_UI
    try:
        _SIMULATOR_UI.close()
    except NameError:
        pass
    except Exception:
        pass

    mode_initial = "Variables priorizadas" if variables_priorizadas_disponibles else "Todas las variables del modelo"
    mode_widget = widgets.ToggleButtons(
        options=["Variables priorizadas", "Todas las variables del modelo"],
        value=mode_initial,
        description="Modo",
    )
    variable_widget = widgets.Dropdown(
        options=variable_options_for_mode(mode_initial),
        description="Variable",
    )
    circuitos = circuit_options()
    circuito_default = circuito_interes_default if circuito_interes_default in circuitos else "Todos"
    circuito_widget = widgets.Dropdown(
        options=circuitos,
        value=circuito_default,
        description="CIRCUITO",
    )
    vano_widget = widgets.Dropdown(
        options=vano_options_for_circuit(circuito_default),
        value="Todos",
        description="FID_VANO",
    )
    fecha_inicio_widget = widgets.DatePicker(
        description="Fecha inicio",
        value=default_fecha_inicio,
        disabled=valid_context_dates.empty,
    )
    fecha_fin_widget = widgets.DatePicker(
        description="Fecha fin",
        value=default_fecha_fin,
        disabled=valid_context_dates.empty,
    )
    filter_summary = widgets.HTML()
    value_box = widgets.VBox()
    run_button = widgets.Button(description="Analizar transición", button_style="primary")
    image_widget = widgets.Image(format="png")

    def refresh_variable_options(*_):
        options = variable_options_for_mode(mode_widget.value)
        variable_widget.options = options
        if options:
            variable_widget.value = options[0]

    def refresh_value_widget(*_):
        if not variable_widget.value:
            value_box.children = []
            return
        value_box.children = [make_value_widget(variable_widget.value)]

    def current_mask():
        return filter_mask(
            circuito=circuito_widget.value,
            fid_vano=vano_widget.value,
            fecha_inicio=fecha_inicio_widget.value,
            fecha_fin=fecha_fin_widget.value,
        )

    def refresh_filter_summary(*_):
        n_rows = int(current_mask().sum())
        filter_summary.value = f"<b>Filas dentro del filtro:</b> {n_rows:,}"

    def refresh_vanos(*_):
        options = vano_options_for_circuit(circuito_widget.value)
        previous = vano_widget.value
        vano_widget.options = options
        vano_widget.value = previous if previous in options else "Todos"
        refresh_filter_summary()

    def on_run(_):
        image_widget.value = b""
        variable = variable_widget.value
        if not variable:
            print("No hay variable seleccionada.")
            return
        if fecha_inicio_widget.value and fecha_fin_widget.value and fecha_inicio_widget.value > fecha_fin_widget.value:
            print("La fecha inicio no puede ser posterior a la fecha fin.")
            return
        mask = current_mask()
        if not mask.any():
            print("No hay filas disponibles con los filtros seleccionados.")
            return
        value_control = value_box.children[0] if value_box.children else None
        selected_value = getattr(value_control, "value", None)
        if is_categorical_variable(variable):
            selected_value = None
        values_original = values_grid_for_variable(variable, selected_value)
        try:
            result_df, metadata = simulate_feature_class_transitions(
                model=modelo,
                X_scaled=X,
                X_raw_model=X_raw_model,
                original_feature_df=Xdf,
                feature_names=feature_names,
                variable=variable,
                values_original=values_original,
                feature_scaler=feature_scaler,
                predict_fn=predict_classification,
                device=DEVICE,
                mask=mask,
                label_encoders=label_encoders,
                max_values_imputed=max_values_imputed,
            )
        except Exception as exc:
            print(f"No se pudo simular {variable}: {exc}")
            return
        for message in metadata.get("warnings", []):
            warnings.warn(message, RuntimeWarning)
        if result_df.empty:
            print("La simulacion no produjo filas validas.")
            return
        fig, plot_path = save_simulation_outputs(result_df, variable)
        plt.close(fig)
        image_widget.value = Path(plot_path).read_bytes()

    mode_widget.observe(refresh_variable_options, names="value")
    variable_widget.observe(refresh_value_widget, names="value")
    circuito_widget.observe(refresh_vanos, names="value")
    vano_widget.observe(refresh_filter_summary, names="value")
    fecha_inicio_widget.observe(refresh_filter_summary, names="value")
    fecha_fin_widget.observe(refresh_filter_summary, names="value")
    run_button.on_click(on_run)
    refresh_value_widget()
    refresh_filter_summary()

    top_row = widgets.HBox([mode_widget, variable_widget])
    filter_row = widgets.HBox([circuito_widget, vano_widget])
    date_row = widgets.HBox([fecha_inicio_widget, fecha_fin_widget, filter_summary])
    ui = widgets.VBox([
        top_row,
        filter_row,
        date_row,
        value_box,
        run_button,
        image_widget,
    ])
    _SIMULATOR_UI = ui
    display(ui)
    return None


## 5. Interfaz interactiva

El modo inicial es `Variables priorizadas` si existe una tabla válida. Usa únicamente `CIRCUITO`, `FID_VANO` y el rango de fecha para definir el subconjunto; el análisis se hace sobre todas las filas dentro de esos filtros. En variables categóricas se evalúan automáticamente todas las categorías disponibles.


In [11]:
clear_output(wait=True)
_ = interactive_classification_feature_simulator()
